# 第 29 章 分析の検証と改訂

Book 第 29 章に対応します。前章の検証で約 4 MPa だった森の誤差について，条件を決めてから保留の記録で確かめます。同じ範囲で基準と森を比べることと，同じ森を異なる記録で評価することを分けて読みます。

前半は講義の実演です。後半は約 20 分で，前回の自分の計画に書いた確認を実行し，結果に合わせて説明を更新します。上から順に実行してください。

## 決めていた条件

| 何を使うか | 今回の条件 |
|---|---|
| 学習 | 631 行・256 配合。八つの入力から森を学習し，強度の平均を基準にする。 |
| 選択に使った検証 | 208 行・86 配合。前章の比較を同じ規則で読み直す。 |
| 手順を決めた後の保留 | 191 行・86 配合。条件を決めた後に基準と森を適用する。 |
| 主な指標 | 全行を一つずつ等しく数える MAE（MPa）。 |
| 追加の確認 | 配合ごとの平均を等しく数えた値。材齢 1〜7 日，8〜28 日，29 日以上の件数と MAE，範囲の比率をそろえた補助の値。 |

検証で方法を選んだ後も，同じ学習用の 631 行から作った森と基準を使います。森は 200 本，`max_features=0.7`，乱数の種は 2026 です。測定前に分かる七つの材料量と指定材齢を入力し，評価側の実測強度は採点のときに照合します。元の表は公開されています。保留とは，ここまでの教材の方法選択に使わず残した記録のことです。

In [ ]:
from pathlib import Path

from IPython.display import display
import pandas as pd
from sklearn.ensemble import RandomForestRegressor

csv_paths = [
    Path("data/concrete-strength.csv"),
    Path("../data/concrete-strength.csv"),
    Path("concrete-strength.csv"),
]
local_csv = next((path for path in csv_paths if path.is_file()), None)
public_csv = (
    "https://raw.githubusercontent.com/sohei-takagi/"
    "python-ds-course-materials/main/data/concrete-strength.csv"
)
concrete = pd.read_csv(local_csv if local_csv is not None else public_csv)
train = concrete.loc[concrete["split"] == "train"]
valid = concrete.loc[concrete["split"] == "validation"]
holdout = concrete.loc[concrete["split"] == "holdout"]

features = [
    "cement", "slag", "fly_ash", "water", "superplasticizer",
    "coarse_aggregate", "fine_aggregate", "age",
]
forest = RandomForestRegressor(
    n_estimators=200, max_features=0.7, random_state=2026, n_jobs=2,
)
forest.fit(train[features], train["strength"])
baseline_strength = train["strength"].mean()

## 同じ行で比べる

一行ごとに実測と予測を並べます。絶対誤差は，二つの差を取ってから `abs()` で大きさにした値です。保留の元行 2 と元行 7 では，どちらの方法の絶対誤差が小さいかをそれぞれ確かめます。

検証と保留のそれぞれで，全行の誤差を一行ずつ同じ重さで平均します。保留の行で基準と森を比べ，次に森だけを検証と保留で比べます。それぞれが，どの予想を確かめる比較か説明してください。

In [ ]:
def evaluate_rows(rows):
    result = rows[["record_id", "recipe_id", "age", "strength"]].copy()
    result["baseline_prediction"] = baseline_strength
    result["forest_prediction"] = forest.predict(rows[features])
    result["baseline_error"] = (result["strength"] - result["baseline_prediction"]).abs()
    result["forest_error"] = (result["strength"] - result["forest_prediction"]).abs()
    return result

valid_errors = evaluate_rows(valid)
holdout_errors = evaluate_rows(holdout)
holdout_errors[
    ["record_id", "age", "strength", "forest_prediction", "baseline_error", "forest_error"]
].head(3).round(3)

In [ ]:
error_columns = ["baseline_error", "forest_error"]
evaluations = {"Validation": valid_errors, "Holdout": holdout_errors}

row_mae = pd.DataFrame({
    name: rows[error_columns].mean()
    for name, rows in evaluations.items()
}).T
row_mae.round(2)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, ax = plt.subplots(figsize=(7.2, 3.9), layout="constrained")
x = np.arange(len(row_mae))
for offset, column, label, color, hatch in [
    (-0.19, "baseline_error", "Training mean", "#a35838", "/"),
    (0.19, "forest_error", "Random forest", "#17667f", ""),
]:
    bars = ax.bar(x + offset, row_mae[column], width=0.35, label=label,
                  color=color, edgecolor="#374149", hatch=hatch)
    ax.bar_label(bars, fmt="%.2f", padding=4)
ax.set(xticks=x, xticklabels=[
    f"{name}\n(n = {len(rows)} rows)" for name, rows in evaluations.items()
], ylabel="MAE (MPa)", ylim=(0, 17.8))
ax.set_axisbelow(True)
ax.grid(axis="y", color="#d9dfe2", linewidth=0.7)
ax.legend(loc="upper left", ncol=2)
plt.show()

配合を等しく扱う場合は，各配合の中で先に平均を取り，次に配合どうしを平均します。同じ配合の同じ材齢に複数行があれば，その複数行も最初の平均に使います。保留の森について，191 行の平均と 86 配合の平均のどちらが主と決めた値か確かめます。

In [ ]:
recipe_mae = pd.DataFrame({
    name: rows.groupby("recipe_id")[error_columns].mean().mean()
    for name, rows in evaluations.items()
}).T
recipe_mae.round(2)

## 材齢の範囲へ戻る

`1to7` は 1〜7 日，`8to28` は 8〜28 日，`29plus` は 29 日以上です。右端を含む範囲を指定し，行数，配合数，二方法の MAE を出します。一配合の行が複数の範囲に入ることはあります。行がない範囲は件数だけが 0 で，MAE を 0 MPa として数えません。

左の図は行の割合，右はその範囲の森の MAE です。左の図で割合が近い範囲でも，右の図では誤差が異なり得ます。29 日以上の棒を両方の図で確認します。

In [ ]:
age_bands = ["1to7", "8to28", "29plus"]

def summarize_bands(rows):
    age_group = pd.cut(
        rows["age"], bins=[0, 7, 28, float("inf")], labels=age_bands, right=True,
    )
    counts = rows.groupby(age_group, observed=True).agg(
        Rows=("record_id", "size"),
        Recipes=("recipe_id", "nunique"),
        Baseline_MAE=("baseline_error", "mean"),
        Forest_MAE=("forest_error", "mean"),
    ).reindex(age_bands)
    counts[["Rows", "Recipes"]] = counts[["Rows", "Recipes"]].fillna(0).astype(int)
    return counts

valid_bands = summarize_bands(valid_errors)
holdout_bands = summarize_bands(holdout_errors)
pd.concat({"Validation": valid_bands, "Holdout": holdout_bands}).round(2)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 2, figsize=(9.1, 4.15), layout="constrained")
positions = np.arange(len(age_bands))
for name, bands, offset, color, hatch in [
    ("Validation", valid_bands, -0.17, "#627e8b", ""),
    ("Holdout", holdout_bands, 0.17, "#a85934", "/"),
]:
    for axis, values in zip(axes, [100 * bands["Rows"] / bands["Rows"].sum(), bands["Forest_MAE"]]):
        axis.bar(positions + offset, values, width=0.32, label=name,
                 color=color, edgecolor="#3b454b", hatch=hatch)
for axis in axes:
    axis.set(xticks=positions, xticklabels=["1–7 days", "8–28 days", "29+ days"])
    axis.set_axisbelow(True)
    axis.grid(axis="y", color="#d9dfe2", linewidth=0.7)
axes[0].set(ylabel="Share of rows (%)", ylim=(0, 64))
axes[1].set(ylabel="Forest MAE (MPa)", ylim=(0, 10.2))
fig.legend(*axes[0].get_legend_handles_labels(), loc="outside upper center", ncols=2)
plt.show()

## 全行の平均に戻す

範囲の MAE にその範囲の行数を掛けると，絶対誤差の合計に戻ります。保留の三つの合計を足し，191 行で割ります。各範囲を一つずつ同じ重さで平均する計算と，何が違うか説明できますか。

次に保留の範囲内 MAE は保ち，割合を検証の `52/208`，`99/208`，`57/208` にします。`common_mix_mae` は保留の誤差を検証の割合で組み合わせた値です。実際に採点した全保留行の MAE は 5.55 MPa です。どちらかの評価表で範囲に行がない場合，共通構成の値は計算できません。

図の三本について，何の記録から計算し，どの割合を使ったかを区別します。材齢範囲の割合だけで，全体の違いを説明できるでしょうか。

In [ ]:
holdout_present = holdout_bands.loc[holdout_bands["Rows"] > 0]
error_totals = holdout_present["Forest_MAE"] * holdout_present["Rows"]
overall_from_bands = error_totals.sum() / len(holdout_errors)

holdout_breakdown = holdout_present[["Rows", "Forest_MAE"]].assign(
    Total_abs_error=error_totals,
)
print(f"Mean from the band totals: {overall_from_bands:.2f} MPa")
holdout_breakdown.round(2)

In [ ]:
mae_columns = ["Baseline_MAE", "Forest_MAE"]
validation_weights = valid_bands["Rows"] / len(valid_errors)
common_mix_mae = pd.Series(float("nan"), index=mae_columns)

if (valid_bands["Rows"] > 0).all() and (holdout_bands["Rows"] > 0).all():
    common_mix_mae = holdout_bands[mae_columns].mul(validation_weights, axis="index").sum()
else:
    print("An age range has no rows; a common-composition mean cannot be calculated.")

common_mix_mae.round(2)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(7.5, 3.4), layout="constrained")
bars = ax.barh(
    ["Validation · observed rows", "Holdout · observed rows", "Holdout · validation mix"],
    [row_mae.loc["Validation", "forest_error"],
     row_mae.loc["Holdout", "forest_error"], common_mix_mae["Forest_MAE"]],
    color=["#6f8792", "#17667f", "#e9dcce"], edgecolor="#3b454b", height=0.57,
)
bars[2].set_hatch("//")
ax.bar_label(bars, fmt="%.2f", padding=4)
ax.set(xlim=(0, 7.25), xlabel="Forest MAE (MPa)")
ax.invert_yaxis()
ax.set_axisbelow(True)
ax.grid(axis="x", color="#d9dfe2", linewidth=0.7)
plt.show()

## 説明の更新と再実行

保留で森は基準より近くなりました。森の誤差は検証 3.97 MPa に対して保留 5.55 MPa で，29 日以上では 8.37 MPa でした。「検証では約 4 MPa」という説明を残し，保留でも同程度だったと言えるかを考えます。範囲を区切った結果だけから誤差の原因を決めず，次に試験する材齢や配合，記録する測定条件を挙げます。

同じ保留の列から MAE を再計算し，最初の出力と照合します。同じ手順の再実行で数値がそろったことと，別の記録で性能を確かめたことを区別します。

In [ ]:
from math import isclose

prediction_again = forest.predict(holdout[features])
mae_again = (holdout["strength"] - prediction_again).abs().mean()
print(f"Holdout: {len(holdout)} rows / {holdout['recipe_id'].nunique()} recipes")
print(f"MAE from the original columns: {mae_again:.2f} MPa")
print("Matches the evaluation table:", isclose(mae_again, row_mae.loc["Holdout", "forest_error"]))

## 実習（約 20 分）

### 1．前回の計画を読む（約 4 分）

自分の作業ノートを開き，第 28 章のあとに残した問い，範囲，方法，指標と予想を読みます。**前回の予想を保存したまま**，今日どの結果を読み直すか書きます。

| 前回決めた確認 | 次のセルで使う値 |
|---|---|
| 指定の材齢範囲の予測誤差を確かめる | `question_kind = "prediction"`。保存した `chosen_band` を `"1to7"`，`"8to28"`，`"29plus"` から指定する。 |
| 二材齢の強度の差を記述する | `question_kind = "description"`。保存した二つを `chosen_ages` に指定する。平均と中央値は `"median"`，分布と正負の件数は `"signs"` を `description_check` に指定する。 |
| 別の既習題材で決めた確認を行う | `question_kind = "other"`。保存したノートでその確認を実行する。 |

**前回の計画・予想と，今回照合する出力**　（ここに記入）

### 2．決めた条件で確かめる（約 7 分）

予測の場合は，保留の行だけを持つ `holdout_errors` から，`lower_age` と `upper_age` の両端を含む行を選びます。`selected_holdout = None` の右辺を `.loc[]` と `.between()` などで書き換えてください。検証の確認表は，同じ材齢範囲について表示します。

記述の場合は，材齢 3・7・28・56 日から異なる二つを選んだ計画を使います。学習表で配合と材齢ごとに平均し，両材齢のそろう配合だけを比べます。差の向きは遅い材齢 − 早い材齢です。下の準備セルは，前回選んだ確認を同じ単位と範囲で出す関数です。

In [ ]:
def check_description(source, chosen_ages, description_check):
    early, late = sorted(chosen_ages)
    records = source.loc[source["age"].isin([early, late])]
    recipe_age = records.groupby(["recipe_id", "age"], as_index=False)["strength"].mean()
    by_recipe = recipe_age.pivot(index="recipe_id", columns="age", values="strength")
    paired = by_recipe.reindex(columns=[early, late]).dropna()
    differences = paired[late] - paired[early]
    if differences.empty:
        print("両方の材齢がそろう配合がないため，差は求められません。")
        return

    print(f"学習表・材齢 {early} 日 → {late} 日: {len(differences)} 配合")
    if description_check == "median":
        display(differences.agg(["mean", "median"]).rename("Difference (MPa)").round(2))
    else:
        import matplotlib.pyplot as plt

        display(pd.Series({
            "Positive": differences.gt(0).sum(),
            "Zero": differences.eq(0).sum(),
            "Negative": differences.lt(0).sum(),
        }, name="Recipes"))
        fig, ax = plt.subplots(figsize=(6.9, 3.55), layout="constrained")
        ax.hist(differences, bins=8, color="#17667f", edgecolor="white")
        ax.axvline(0, color="#555b63", linewidth=1.1, linestyle="--")
        ax.set(title=f"{early} to {late} days · {len(differences)} paired recipes",
               xlabel="Later minus earlier strength (MPa)", ylabel="Recipes")
        plt.show()

In [ ]:
question_kind = None      # "prediction", "description", or "other"
chosen_band = None        # "1to7", "8to28", "29plus"
chosen_ages = None        # e.g., [3, 28] using two distinct ages from 3, 7, 28, 56
description_check = None  # "median" or "signs"

if question_kind == "prediction":
    age_limits = {"1to7": (1, 7), "8to28": (8, 28), "29plus": (29, float("inf"))}
    if chosen_band not in age_limits:
        print("前回選んだ 1to7・8to28・29plus のいずれかを指定してください。")
    else:
        lower_age, upper_age = age_limits[chosen_band]
        selected_holdout = None  # holdout_errors.loc[自分で書いた行の条件]
        expected_rows = holdout_errors["age"].between(lower_age, upper_age)
        if selected_holdout is None:
            print("selected_holdout の右辺に，保留の該当行を取り出す式を書いてください。")
        elif not isinstance(selected_holdout, pd.DataFrame):
            print("保留の該当行を表として選んでください。")
        elif not selected_holdout.index.equals(holdout_errors.index[expected_rows]):
            print("選んだ範囲に入る保留の全行を，元の順で選んでください。")
        elif selected_holdout.empty:
            print("範囲に保留の行がないため，MAE は求められません。")
        else:
            scope_result = summarize_bands(selected_holdout).loc[chosen_band]
            scope_table = pd.DataFrame({
                "Validation": valid_bands.loc[chosen_band],
                "Holdout": scope_result,
            }).T
            scope_table[["Rows", "Recipes"]] = scope_table[["Rows", "Recipes"]].astype(int)
            print("事前に選んだ材齢範囲", chosen_band)
            display(scope_table.round(2))
elif question_kind == "description":
    allowed_ages = [3, 7, 28, 56]
    if (not isinstance(chosen_ages, (list, tuple)) or len(chosen_ages) != 2
            or any(day not in allowed_ages for day in chosen_ages)
            or chosen_ages[0] == chosen_ages[1]):
        print("前回選んだ異なる二材齢を chosen_ages に指定してください。")
    elif description_check not in ["median", "signs"]:
        print("前回決めた確認を median または signs で指定してください。")
    else:
        check_description(train, chosen_ages, description_check)
elif question_kind == "other":
    print("前回のノートを開き，結果を見る前に決めた手順を実行してください。")
else:
    print("question_kind に prediction・description・other のいずれかを入れてください。")

### 3．説明を更新する（約 6 分）

- **残す説明**　予想のどの部分を，今回の結果でも説明できますか。何の表のどの値が根拠かを書きます。
- **改める説明**　予測なら，保留の同じ行での基準との比較と，同じ範囲での検証からの変化を分けます。記述なら，平均と追加の要約から，どのような配合の違いを書き添えるかを考えます。
- **次の比較**　どの配合・材齢・測定条件を新しく調べると，説明を確かめられますか。結果を見て新しく思いついた確認は，先に決めた確認と区別してメモします。

**今回の結果，改めた説明と次の比較**　（ここに記入）

講義で読んだ保留の範囲や，第 27 章で読んだ 7 日・28 日の結果を扱う場合は，前に見た値を自分の問いに当てて確かめたと書きます。記述の標準ルートは，同じ学習表の別のまとめ方を読む確認です。

### 4．再実行と保存を点検する（約 3 分）

前回までの作業ノートに，今回の条件，実行したセル，結果，書き直した説明を追加します。予測の計算を移す場合は，森を学習するセル，誤差を作るセル，材齢範囲を集計するセル，実習のセルをその順に置きます。実習のセルより前に，表を表示するための `display` と `pandas` を読み込んでください。記述の場合は，学習表を読んだセルの後に準備セルと実習セルを置きます。

選択を読み直してセルを実行したら，出力の材齢・行数・配合数を確かめます。前回の予想を残し，自分のノートを上から再実行できる形で保存します。

### データの出典

Yeh, I. (1998). *Concrete Compressive Strength*. UCI Machine Learning Repository. Copyright Prof. I-Cheng Yeh. 教材は列名を短縮し，照合と分割に用いる列を追加しています。 [UCI データ DOI](https://doi.org/10.24432/C5PK67) ・ [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/) ・ [教材データの出典と加工](https://github.com/sohei-takagi/python-ds-course-materials/blob/main/data/concrete-sources.md)